# Chapter 22
## A Wilson-Cowan Model of an Oscillatory E-I Network
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter22.ipynb)

## About this chapter

Wilson-Cowan equations replace individual membrane voltages with
firing-rate variables for excitatory and inhibitory populations:
$\tau_E\dot E=-E+f(w_{EE}E-w_{IE}I+I_E)$,
$\tau_I\dot I=-I+g(w_{EI}E-w_{II}I+I_I)$. Recurrent excitation and delayed
inhibitory feedback make a population rhythm, visible in rate traces, a
phase plane, and a rastergram. `simulate_wilson_cowan_e_and_i` plots the
E/I rate traces; `simulate_wilson_cowan_lowering_w_ee` shows how lowering
recurrent excitation $w_{EE}$ changes the long-time E-I trajectory;
`plot_wilson_cowan_phase_plane` draws the E-I nullclines and flow field;
`simulate_wilson_cowan_rastergram` renders the same rate trajectory as a
population-style spike raster via Poisson thinning.

See [`README.md`](chapter22.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from ipywidgets import interact

## E/I Response Functions (shared by every example below)

In [ ]:
def f(x):
    return 100.0 * x ** 2 / (900 + x ** 2) * (x > 0)


def g(x):
    return 100.0 * x ** 2 / (400 + x ** 2) * (x > 0)


def wilson_cowan_derivative(x, t, w_EE=1.5, w_IE=1.0, w_EI=1.0, w_II=0.0,
                             tau_E=5.0, tau_I=10.0, I_E=20.0, I_I=0.0):
    E, I = x
    dE = (f(w_EE * E - w_IE * I + I_E) - E) / tau_E
    dI = (g(w_EI * E - w_II * I + I_I) - I) / tau_I
    return [dE, dI]

## Excitatory and Inhibitory Rate Traces

In [ ]:
def simulate_wilson_cowan_e_and_i(w_EE=1.5, w_IE=1.0, w_EI=1.0, w_II=0.0,
                                   tau_E=5.0, tau_I=10.0, I_E=20.0, I_I=0.0,
                                   t_final=300.0, dt=0.01, E0=50.0, I0=10.0):
    t = np.arange(0, t_final, dt)
    sol = odeint(wilson_cowan_derivative, [E0, I0], t,
                 args=(w_EE, w_IE, w_EI, w_II, tau_E, tau_I, I_E, I_I))
    return t, sol[:, 0], sol[:, 1]


def plot_wilson_cowan_e_and_i(t, E, I):
    plt.figure(figsize=(7, 3))
    plt.plot(t, E, lw=2, c="r", label="E")
    plt.plot(t, I, lw=2, c="b", label="I")
    plt.xlim(min(t), max(t))
    plt.ylim(0, 100)
    plt.xlabel("time [ms]", fontsize=14)
    plt.ylabel("v [mV]", fontsize=14)
    plt.yticks([0, 50, 100])
    plt.tick_params(labelsize=14)
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wilson_cowan_e_and_i(*simulate_wilson_cowan_e_and_i())

## Lowering Recurrent Excitation $w_{EE}$

A fixed-step Heun integration (rather than `odeint`) of the same system,
run at three values of $w_{EE}$ to show how weaker recurrent excitation
changes the long-time E-I trajectory -- plotted in the $(E,I)$ phase plane
after discarding the initial transient.

In [ ]:
def simulate_wilson_cowan_lowering_w_ee(w_EE, w_IE=1.0, w_EI=1.0, w_II=0.0,
                                         tau_E=5.0, tau_I=10.0, I_E=20.0, I_I=0.0,
                                         t_final=1000.0, dt=0.01, E0=10.0, I0=10.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    E = np.zeros(m_steps + 1)
    I = np.zeros(m_steps + 1)
    E[0], I[0] = E0, I0
    for k in range(m_steps):
        E_inc = (f(w_EE * E[k] - w_IE * I[k] + I_E) - E[k]) / tau_E
        I_inc = (g(w_EI * E[k] - w_II * I[k] + I_I) - I[k]) / tau_I
        E_tmp = E[k] + dt05 * E_inc
        I_tmp = I[k] + dt05 * I_inc
        E_inc = (f(w_EE * E_tmp - w_IE * I_tmp + I_E) - E_tmp) / tau_E
        I_inc = (g(w_EI * E_tmp - w_II * I_tmp + I_I) - I_tmp) / tau_I
        E[k + 1] = E[k] + dt * E_inc
        I[k + 1] = I[k] + dt * I_inc
    return E, I


def plot_wilson_cowan_lowering_w_ee(panels):
    fig, ax = plt.subplots(1, 3, figsize=(12, 4.5))
    for a, (w_EE, E, I) in zip(ax, panels):
        a.plot(E, I, '-k', linewidth=2)
        a.set_xlim(0, 100)
        a.set_ylim(0, 100)
        a.set_box_aspect(1)
        a.set_xlabel('$E$')
        a.set_title(f'$w_{{EE}}={w_EE}$')
    ax[0].set_ylabel('$I$')
    plt.tight_layout()
    plt.show()

In [ ]:
w_EE_vec = [1.5, 1.25, 1.0]
panels = []
for w_EE in w_EE_vec:
    E, I = simulate_wilson_cowan_lowering_w_ee(w_EE)
    m_steps = len(E) - 1
    ind = slice(round(4 * m_steps / 5) - 1, m_steps + 1)  # matlab's ind is 1-indexed
    panels.append((w_EE, E[ind], I[ind]))
plot_wilson_cowan_lowering_w_ee(panels)

## E-I Phase Plane

The flow field and nullclines ($\dot E=0$, $\dot I=0$, found by bisection)
in the $(E,I)$ rate phase plane.

In [ ]:
def plot_wilson_cowan_streamplot(ax, w_EE=1.5, w_IE=1.0, w_EI=1.0, w_II=0.0,
                                  tau_E=5.0, tau_I=10.0, I_E=20.0, I_I=0.0,
                                  x=(0, 100), y=(0, 100), dx=0.1, dy=0.1):
    phi1, phi2 = np.meshgrid(np.arange(x[0], x[1], dx), np.arange(y[0], y[1], dy))
    dphi1_dt, dphi2_dt = wilson_cowan_derivative(
        [phi1, phi2], 0, w_EE, w_IE, w_EI, w_II, tau_E, tau_I, I_E, I_I)
    ax.streamplot(phi1, phi2, dphi1_dt, dphi2_dt, color='k', linewidth=0.5, cmap=plt.cm.autumn)


def _nullcline(deriv, ax, label=None, color="r"):
    """Bisects deriv(E, I) over I for each E to trace deriv(E, I)=0."""
    x_max = 100.0
    I_left = np.arange(0, x_max, 0.1)
    I_right = np.arange(0.1, x_max + 0.1, 0.1)
    nx = len(I_left)
    E_red, I_red = [], []
    for i in range(nx):
        E = i / float(nx) * 100.0
        R_left = deriv(E, I_left)
        R_right = deriv(E, I_right)
        ind = np.where((R_left * R_right) < 0)[0]
        for j in ind:
            I_l, I_r = I_left[j], I_right[j]
            while (I_r - I_l) > 1e-8:
                I_c = 0.5 * (I_r + I_l)
                R_l = deriv(E, I_l)
                R_c = deriv(E, I_c)
                if (R_l * R_c) < 0:
                    I_r = I_c
                else:
                    I_l = I_c
                I_c = 0.5 * (I_r + I_l)
                E_red.append(E)
                I_red.append(I_c)
    ax.plot(E_red, I_red, lw=2, c=color, ls="--", label=label)
    if label:
        ax.legend()


def plot_wilson_cowan_phase_plane(w_EE=1.5, w_IE=1.0, w_EI=1.0, w_II=0.0,
                                   tau_E=5.0, tau_I=10.0, I_E=20.0, I_I=0.0):
    def dE(E, I):
        return (f(w_EE * E - w_IE * I + I_E) - E) / tau_E

    def dI(E, I):
        return (g(w_EI * E - w_II * I + I_I) - I) / tau_I

    fig, ax = plt.subplots(figsize=(5, 5))
    plot_wilson_cowan_streamplot(ax, w_EE, w_IE, w_EI, w_II, tau_E, tau_I, I_E, I_I)
    _nullcline(dE, ax, label="dE/dt=0", color="r")
    _nullcline(dI, ax, label="dI/dt=0", color="b")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel("E", fontsize=14)
    ax.set_ylabel("I", fontsize=14)
    ax.tick_params(labelsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wilson_cowan_phase_plane()

## Rastergram

Reads the continuous E/I rate trajectory above as a Poisson firing rate
per neuron (thinning: a spike occurs in $[t,t+dt)$ with probability
$\propto$ rate) to render a population-style spike raster from 80
excitatory and 20 inhibitory "neurons".

In [ ]:
def _extract_spikes(x, t, num_nodes, dt):
    spikes = []
    num_steps = len(x)
    for m in range(num_nodes):
        spk = []
        for i in range(num_steps - 1):
            r = np.random.rand()
            if r < 5e-4 * (x[i] + x[i + 1]) * dt:
                spk.append(t[i])
        spikes.append(spk)
    return spikes


def simulate_wilson_cowan_rastergram(num_e=80, num_i=20, dt=0.01, **kwargs):
    t, E, I = simulate_wilson_cowan_e_and_i(dt=dt, **kwargs)
    spikes_e = _extract_spikes(E, t, num_e, dt)
    spikes_i = _extract_spikes(I, t, num_i, dt)
    return t, spikes_e, spikes_i


def plot_wilson_cowan_rastergram(t, spikes_e, spikes_i):
    num_e, num_i = len(spikes_e), len(spikes_i)
    fig, ax = plt.subplots(1, figsize=(7, 4))
    for ii in range(num_e):
        ax.plot(spikes_e[ii], [ii + num_i] * len(spikes_e[ii]), '.', c='r', markersize=2)
    for ii in range(num_i):
        ax.plot(spikes_i[ii], [ii] * len(spikes_i[ii]), '.', c='b', markersize=2)
    ax.set_xlim(min(t), max(t))
    ax.set_ylim(0, num_e + num_i)
    ax.set_xlabel("time [ms]", fontsize=14)
    ax.set_ylabel("v [mV]", fontsize=14)
    plt.tick_params(labelsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wilson_cowan_rastergram(*simulate_wilson_cowan_rastergram())